Collecting the gold label data:

In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os 

# keys.env with stored keys required 
key_path = '../keys.env'
load_dotenv(dotenv_path=key_path)
api_key = os.getenv("API_KEY")
client = OpenAI(api_key=api_key)

def ask_gpt(prompt):
    prompt = prompt.replace('\x00', '')
    prompt = ''.join(ch for ch in prompt if ord(ch) >= 32 or ch in '\n\r\t')

    completion = client.chat.completions.create(
        model="gpt-5",
        messages=[
            {"role": "assistant", "content": "You are an expert in academic writing and citation analysis."},
            {
                "role": "user",
                "content": prompt
            }
        ],
        reasoning_effort="minimal", 
    )

    return completion.choices[0].message.content

### GPT span classifications
- filter dataset to remove 'References' 
- create prompt for zero and few shot prompting
- collect GPT responses

checklist prompting

Prompts:

In [2]:
format_def = """Issues in citation formatting such as a missing bracket and using the wrong style of citing.
    a) Due to preprocessing errors of the source dataset, some words contain hyphens that do not require it, and some are missing hyphens where it is required. Please ignore these types of formatting issues.
    b) Highlight the word/citation in which the formatting issue occurs in and not only the issue within the word/citation.
    c) Formatting issues appear as either citations or parts of a citation.
    Examples of formatting issues include:
        i) Narrative citation missing year: “Vatswani et al.” -> should be “Vatswani et al. (2020)”
        ii) Wrong citation style: “In (Vatswani et al., 2019)” -> should be “in Vatswani et al. (2019)”
        iii) Wrong use of footnotes: "Vastwani et al. 1" -> should include the year or be reformatted as a proper footnote."""

unsupp_def = """claim about prior work or statistics w/o citation or evidence. 
    a) The author should cite at every first mention of a study, paper, shared task, competition or dataset.
    b) Specific information to a niche topic, despite sounding like it should be known in that topic of study, should be cited.
    c) If a claim is made and is obvious to be a natural deduction from previous statements through common sense (i.e not requiring expert knowledge), then this claim does not fall under ‘Unsupported claim’. For example:
        i) “However, creating a large and suitable set of questions for supporting narrative comprehension is both time-consuming and cognitively demanding.” -> it is obvious that creating a dataset is time consuming and mentally demanding.
    d) Any mention of “recent works” should be backed up with citations to the works.
    e) Unsupported claim issues appear as single words, phrases (ex. 'BERT'), sub-sentences or full sentences.
    Examples of unsupported claims include:
        i) Missing citations for mentions of 'recent works': “and there are many recent works that explore this topic”,
        ii) Mention of a previous work and claim without citation: “..., while in a previous study, the authors claim …”,
        iii) Mentioning of a specific setup of a task without citation to the work: ".. BERT was used in an AES task trained on essays .." """

lacksynth_def = """occurs when either:
    a) The author describes or cites papers without connecting them to their own work/argument 
    b) Or only follows up the summary of previous works with their own contribution without explicitly highlighting the gap their work intends to research.
    c) It does not articulate the author's perspective or motivation.
    d) A lack of argument/opinion in the first paragraph is permissible as it serves to be the foundation of the author's argument 
    e) Lacks synthesis issues appear either as single sentences or multiple sentences.
    Examples of lack of synthesis include:
        i) No elaboration of own contribution/argument:"Following early neural approaches to question answering, many subsequent studies adopt a pipeline architecture consisting of retrieval and comprehension components. The retrieval component focuses on identifying relevant documents or passages from a large corpus, while the comprehension component extracts an answer span from the retrieved text. Initial models relied on recurrent neural networks with attention mechanisms to encode questions and contexts (Seo et al., 2017; Wang et al., 2017)."
        ii)  No explanation of the cited works and relation to their own work: “Recently, several studies have explored the use of prompting techniques with pre-trained language models to influence model outputs or access latent knowledge (Brown et al., 2020; Gao et al., 2021; Liu et al., 2021; Wei et al., 2022).” """

coherence_def = """connection between cited works is abrupt, lacking relation to each other. It is unclear how one mentioned work is relevant to a prior mentioned work. 
    a) Sentences are not transitioned from one to another.
    b) The relationship between sentences describing papers is implied but not explicitly stated.
    c) Coherence issues appear only as multiple sentences.
    Examples of coherence issues include:
        i) Relation between mentioned works is not explicit: “Smith (2020) identified a relationship between personal belief systems and ethical decision-making frameworks. Moral foundation theory proposes several core dimensions of moral reasoning, including harm, fairness, and authority (Jones, 2015). Audience adaptation has been explored in computational argumentation. Lee et al. (2019) applied moral categories to argument generation tasks. Human annotators often disagree when labeling moral dimensions in text (Nguyen et al., 2018).”
        ii) Lack of transitions between sentences: “Recent studies have explored various techniques for enhancing model performance. Smith et al. (2020) introduced a novel architecture that significantly improves accuracy on benchmark datasets. Additionally, Johnson and Lee (2019) proposed a data augmentation method that increases training data diversity.” 
        iii) No explanation of the cited works and relation to their own work: “Recently, several studies have explored the use of prompting techniques with pre-trained language models to influence model outputs or access latent knowledge (Brown et al., 2020; Gao et al., 2021; Liu et al., 2021; Wei et al., 2022).” """

In [39]:
import json 
import re

def complete_prompt(text, full_text):
    final = text + f"""\n
Respond ONLY in this json format:
{{  "span_texts": [..., ...] ,
    "gpt_label": ... }}
- Please have 'span_text' ONLY contain text. The text must be verbatim from the passage of text provided.
- If you dont find any spans, leave the list empty and have "gpt_label" say 'None'. 

Here is the passage of text you must identify spans in:
{full_text}"""
    return final 

def parse_list(response):
    s = re.sub(r"```(?:json)?\s*", "", response).replace("```", "").strip()

    start_candidates = [i for i in (s.find("["), s.find("{")) if i != -1]
    if not start_candidates:
        raise ValueError("No JSON start token found ('[' or '{').")

    start = min(start_candidates)
    s = s[start:].lstrip()

    decoder = json.JSONDecoder()
    obj, _end = decoder.raw_decode(s)

    if isinstance(obj, dict):
        obj = [obj]

    if not isinstance(obj, list):
        raise ValueError(f"Expected a JSON list, got {type(obj).__name__}")

    if not all(isinstance(item, dict) for item in obj):
        bad_types = {type(item).__name__ for item in obj if not isinstance(item, dict)}
        raise ValueError(f"Expected list of dicts, but found non-dict items: {bad_types}")

    return obj

In [4]:
def extract_start_end_char(text, full_text):
    text_norm = text.replace('.', '')
    full_norm = full_text.replace('.', '')

    if text_norm in full_norm:
        start = full_norm.index(text_norm)
        end = start + len(text_norm)
    else:
        start = end = -1

    return start, end

In [5]:
def sanitize_text(text: str) -> str:
    """Remove characters that break JSON serialization."""
    text = text.replace('\x00', '')
    text = ''.join(ch for ch in text if ord(ch) >= 32 or ch in '\n\r\t')
    return text.strip()

GPT response collection:

In [6]:
from tqdm import tqdm
import time
import spacy
nlp = spacy.load("en_core_web_sm")

prompt_strategy = ['zero', 'fewshot']
zeroshot = []
fewshot = []
real_path = '../synthetic_sampling/samples/total_real_data.json'
categories_def = {
    "Unsupported claim": unsupp_def,
    "Format": format_def,
    "Coherence": coherence_def,
    "Lacks synthesis": lacksynth_def,
}

SENTENCE_LEVEL_CATEGORIES = {"Unsupported claim", "Format"}
PARAGRAPH_LEVEL_CATEGORIES = {"Coherence", "Lacks synthesis"}

MAX_RETRIES = 3
RETRY_DELAY = 2

with open(real_path, 'r') as f:
    real_data = json.load(f)


In [37]:
import os
from pathlib import Path

docs_folder = Path("../../to_annotate")
doc_files = []
skip = [42, 51, 26, 75, 77, 93, 35]

for files in os.listdir(docs_folder):
    if 'paper_' in files:
        number = files.split('_')[1]
        number = int(number.split('.txt')[0])
        if number not in skip:
            with open(os.path.join(docs_folder, files), 'r') as f:
                data = f.read()
            doc = data.split('References')[0]
            doc_files.append(doc)
    
print(f"Found {len(doc_files)} documents")


Found 89 documents


In [ ]:
for i, document in enumerate(tqdm(doc_files, desc="documents")):
    
    fewshot_entry = {"document": document, "categories": {}}
    zeroshot_entry = {"document": document, "categories": {}}

    for category in categories_def.keys():
        definition = categories_def[category]
        use_sentences = category in SENTENCE_LEVEL_CATEGORIES

        if use_sentences:
            doc_nlp = nlp(document)
            units = [sent.text.strip() for sent in doc_nlp.sents if sent.text.strip()]
            unit_label = "Sentence"
        else:
            units = [p.strip() for p in re.split(r'\n\s*\n', document) if p.strip()]
            unit_label = "Paragraph"

        for strategy in prompt_strategy:
            all_spans = []
            seen_units = set()  # track units already sent to GPT to avoid duplicates

            for unit in units:
                unit = sanitize_text(unit)
                if not unit:
                    continue

                # skip duplicate units before calling GPT
                unit_key = unit.strip()[:200]
                if unit_key in seen_units:
                    continue
                seen_units.add(unit_key)

                if strategy == 'fewshot':
                    prompt_template = f"""You are given a {unit_label.lower()} of text that comes from an academic paper on the field of Natural Language Processing. You must highlight all spans that exhibit any of these issues:
**{category}**: {definition}
"""
                else:
                    prompt_template = f"""You are given a {unit_label.lower()} of text that comes from an academic paper on the field of Natural Language Processing. You must highlight all spans that exhibit any of these issues:
**{category}**: {definition.split("Examples of")[0]}
"""

                prompt = complete_prompt(prompt_template, unit)

                for attempt in range(MAX_RETRIES):
                    try:
                        response = ask_gpt(prompt)
                    except Exception as e:
                        print(f"Failed on category={category} doc={i} unit={repr(unit[:100])}: {e}")
                        time.sleep(RETRY_DELAY)
                        continue

                    if len(response) == 0:
                        print(f"[retry {attempt+1}/{MAX_RETRIES}] Empty response for {category} doc={i} ({strategy})")
                        time.sleep(RETRY_DELAY)
                        continue

                    json_response = parse_list(response)

                    unit_spans = []
                    for resp in json_response:
                        span_texts = resp.get('span_texts', [])
                        gpt_label = resp.get('gpt_label', None)

                        if isinstance(span_texts, str):
                            span_texts = [span_texts]

                        if not span_texts:
                            unit_spans.append({
                                **{k: v for k, v in resp.items() if k != 'span_texts'},
                                'gpt_label': gpt_label,
                                'segment_text': unit,
                                'segment_type': unit_label.lower(),
                                'span_text': None,
                                'start': None,
                                'end': None,
                            })
                            continue

                        for span_text in span_texts:
                            if not span_text:
                                unit_spans.append({
                                    **{k: v for k, v in resp.items() if k != 'span_texts'},
                                    'gpt_label': gpt_label,
                                    'segment_text': unit,
                                    'segment_type': unit_label.lower(),
                                    'span_text': None,
                                    'start': None,
                                    'end': None,
                                })
                                continue

                            start, end = extract_start_end_char(span_text, document)
                            unit_spans.append({
                                **{k: v for k, v in resp.items() if k != 'span_texts'},
                                'gpt_label': gpt_label,
                                'segment_text': unit,
                                'segment_type': unit_label.lower(),
                                'span_text': span_text,
                                'start': start,
                                'end': end,
                            })

                    if json_response:
                        all_spans.extend(unit_spans)
                        break

                    print(f"[retry {attempt+1}/{MAX_RETRIES}] No response parsed for {category} doc={i} unit ({strategy})")
                    time.sleep(RETRY_DELAY)

            # deduplicate spans by (segment_text, span_text, start)
            seen = set()
            deduped = []
            for span in all_spans:
                key = (span.get('segment_text'), span.get('span_text'), span.get('start'))
                if key not in seen:
                    seen.add(key)
                    deduped.append(span)

            if strategy == 'fewshot':
                fewshot_entry["categories"][category] = deduped
            else:
                zeroshot_entry["categories"][category] = deduped

    fewshot.append(fewshot_entry)
    zeroshot.append(zeroshot_entry)

    if (i + 1) % 10 == 0:
        with open(f"saves/fewshot_responses_{i+1}.json", "w") as f:
            json.dump(fewshot, f, indent=4)
        with open(f"saves/zero_responses_{i+1}.json", "w") as f:
            json.dump(zeroshot, f, indent=4)

In [27]:
import json 

fewshot_path = "saves/fewshot_preds.json"
zeroshot_path = "saves/zeroshot_preds.json"

with open(fewshot_path, 'w') as f:
    json.dump(fewshot, f, indent=2)

with open(zeroshot_path, 'w') as f:
    json.dump(zeroshot, f, indent=2)


Some spans dont have a start and end index due to GPT over correcting grammar/spelling mistakes, so we do post-processing to fix this:

In [18]:
import re

MISSING_SEMI_SPACE_RE = re.compile(r";\S")  # semicolon not followed by space


def remove_missing_spaces_format_spans(data: list) -> tuple[list, int]:
    """
    Filter Format spans with missing semicolon spaces from an in-memory list of documents.
    Returns the filtered data and the number of removed spans.
    """
    removed = 0

    for doc in data:
        new_gpt_response = []
        for s in doc.get("gpt_response", []):
            if s.get("gpt_label") == "Format":
                reason = (s.get("reason") or "").lower()
                span_text = s.get("span_text") or ""

                if (
                    "missing space" in reason
                    or "missing spaces" in reason
                    or MISSING_SEMI_SPACE_RE.search(span_text)
                ):
                    removed += 1
                    continue

            new_gpt_response.append(s)

        doc["gpt_response"] = new_gpt_response

    print(f"Removed {removed} Format spans for missing semicolon spaces.")
    return data, removed

In [19]:
fewshot, removed_few  = remove_missing_spaces_format_spans(fewshot)
zeroshot, removed_zero = remove_missing_spaces_format_spans(zeroshot)

Removed 15 Format spans for missing semicolon spaces.
Removed 66 Format spans for missing semicolon spaces.


In [22]:
from difflib import SequenceMatcher
import unicodedata
from tqdm import tqdm
def _normalize_with_map(s: str):
    """
    Normalize in a way that often matches annotation tools:
    - Unicode normalize (NFC)
    - Convert CRLF/CR to LF
    - Replace NBSP with space
    - Remove common zero-width chars
    Returns: (normalized_string, norm_index -> original_index map)
    """
    s = unicodedata.normalize("NFC", s)

    norm_chars = []
    norm_to_orig = []

    i = 0
    while i < len(s):
        ch = s[i]

        # normalize newlines
        if ch == "\r":
            if i + 1 < len(s) and s[i + 1] == "\n":
                norm_chars.append("\n")
                norm_to_orig.append(i)
                i += 2
                continue
            else:
                norm_chars.append("\n")
                norm_to_orig.append(i)
                i += 1
                continue

        # normalize NBSP
        if ch == "\u00A0":
            ch = " "

        # drop zero-width characters
        if ch in ("\u200b", "\u200c", "\u200d", "\ufeff"):
            i += 1
            continue

        norm_chars.append(ch)
        norm_to_orig.append(i)
        i += 1

    return "".join(norm_chars), norm_to_orig


def fuzzy_span_start_end_mapped(
    span_text: str,
    full_text: str,
    threshold: float = 0.80,
    window_slack: int = 20,
    output_1_based_inclusive: bool = False,
):
    if not span_text or not full_text:
        return -1, -1

    norm_full, norm_map = _normalize_with_map(full_text)
    norm_span, _ = _normalize_with_map(span_text)

    target_len = len(norm_span)
    best_score = 0.0
    best_start = -1
    best_end = -1

    min_len = max(1, target_len - window_slack)
    max_len = min(len(norm_full), target_len + window_slack)

    for win_len in range(min_len, max_len + 1):
        for i in range(0, len(norm_full) - win_len + 1):
            candidate = norm_full[i:i + win_len]
            score = SequenceMatcher(None, norm_span, candidate).ratio()
            if score > best_score:
                best_score = score
                best_start = i
                best_end = i + win_len

    if best_score < threshold or best_start < 0:
        return -1, -1

    orig_start = norm_map[best_start]
    orig_end = norm_map[best_end - 1] + 1

    if output_1_based_inclusive:
        return orig_start + 1, orig_end

    return orig_start, orig_end


def remove_missing_spaces_format_spans(
    in_path: str,
    out_path: str,
):
    with open(in_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    removed = 0

    for doc in data:
        document = doc.get("document", "")
        new_gpt_response = []

        for s in doc.get("gpt_response", []):
            if s.get("gpt_label") == "Format":
                reason = (s.get("reason") or "").lower()
                span_text = s.get("span_text") or ""

                if (
                    "missing space" in reason
                    or "missing spaces" in reason
                    or MISSING_SEMI_SPACE_RE.search(span_text)
                ):
                    removed += 1
                    continue

                # re-align start/end using fuzzy matching if span_text is present
                if span_text:
                    start, end = fuzzy_span_start_end_mapped(span_text, document)
                    if start != -1:
                        s["start"] = start
                        s["end"] = end

            new_gpt_response.append(s)

        doc["gpt_response"] = new_gpt_response

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

    print(f"Removed {removed} Format spans for missing semicolon spaces.")
    return removed


def fast_span_start_end(span_text: str, full_text: str, threshold: float = 0.80):
    """Try exact match first, fall back to fuzzy only if needed."""
    if not span_text or not full_text:
        return -1, -1

    # fast path: exact match
    idx = full_text.find(span_text)
    if idx != -1:
        return idx, idx + len(span_text)

    # slow path: fuzzy match
    return fuzzy_span_start_end_mapped(span_text, full_text, threshold=threshold)


for resp in tqdm(fewshot, desc='Fuzzy matching few shot samples: '):
    full_text = resp['document']
    for span_obj in resp.get('gpt_response', []):
        span = span_obj.get('span_text', '')
        if not span:
            continue
        s, e = fast_span_start_end(span, full_text)
        span_obj['start'] = s
        span_obj['end'] = e

for resp in tqdm(zeroshot, desc='Fuzzy matching zero shot samples: '):
    full_text = resp['document']
    for span_obj in resp.get('gpt_response', []):
        span = span_obj.get('span_text', '')
        if not span:
            continue
        s, e = fast_span_start_end(span, full_text)
        span_obj['start'] = s
        span_obj['end'] = e


Fuzzy matching zero shot samples: 100%|██████████| 117/117 [02:20<00:00,  1.20s/it]


Save to folder:

In [24]:
import json 

filename_fewshot = "saves/fewshot_preds_final.json"
filename_zeroshot = "saves/zeroshot_preds_final.json"

with open(filename_fewshot, 'w') as f:
    json.dump(fewshot, f, indent=4)

with open(filename_zeroshot, 'w') as f:
    json.dump(zeroshot, f, indent=4)